In [0]:
# ==========================================
# Project: Retail Medallion Architecture
# Layer: Gold
# Purpose: Monthly Revenue KPI
# ==========================================

from pyspark.sql import functions as F

# ------------------------------------------
# Configuration
# ------------------------------------------

SILVER_TABLE = "retail_project.silver.online_retail"
GOLD_TABLE = "retail_project.gold.monthly_revenue"

# ------------------------------------------
# Read Silver Data
# ------------------------------------------

df_silver = spark.table(SILVER_TABLE)

# ------------------------------------------
# Revenue Calculation
# ------------------------------------------

df_gold_base = (
    df_silver
    .withColumn(
        "revenue",
        F.round(
            F.col("Quantity") * F.col("UnitPrice"),
            2
        )
    )
)

# ------------------------------------------
# Monthly Aggregation
# ------------------------------------------

monthly_revenue = (
    df_gold_base
    .withColumn(
        "year_month",
        F.date_format("InvoiceDate", "yyyy-MM")
    )
    .groupBy("year_month")
    .agg(
        F.round(
            F.sum("revenue"),
            2
        ).alias("total_revenue")
    )
    .orderBy("year_month")
)

# ------------------------------------------
# Validation
# ------------------------------------------

monthly_revenue.show(50, False)

# ------------------------------------------
# Write Gold Table
# ------------------------------------------

(
    monthly_revenue.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE)
)

print("Monthly Revenue Gold table created successfully.")